In [ ]:
!pip install datasets

In [12]:
import pandas as pd
import re
from datasets import load_dataset, DownloadConfig

###############################################################################
# 1. READ YOUR EXCEL FILE WITH KEYWORDS
###############################################################################

def load_sdg_keywords_from_excel(xlsx_path):
    """
    Reads an Excel file where each row corresponds to:
      Row columns: [Index, Goal, Target, 0,1,2,3,4,... ] etc.
    The 'Goal' column might be something like 'SDG 1'.
    The subsequent columns contain synonyms for that row.

    Returns a dict: { "SDG 1": set_of_synonyms, "SDG 2": set_of_synonyms, ... }
    """
    df = pd.read_excel(xlsx_path, header=0)
    
    sdg_dict = {}
    for _, row in df.iterrows():
        goal = row["Goal"]  # e.g. 'SDG 1'
        
        # Gather synonyms from columns after index 3 (adjust indexing as needed).
        synonyms = []
        for col_name in df.columns[3:]:
            val = row[col_name]
            if isinstance(val, str) and val.strip():
                synonyms.append(val.strip())
        
        if goal not in sdg_dict:
            sdg_dict[goal] = set()
        
        for syn in synonyms:
            sdg_dict[goal].add(syn.lower())
    
    return sdg_dict

###############################################################################
# 2. RULE-BASED MATCHING
###############################################################################

def match_sdg_keywords(doc_text, sdg_dict):
    """
    doc_text: the text of the document (string).
    sdg_dict: { 'SDG 1': set_of_keywords, 'SDG 2': set_of_keywords, ... }
    
    Returns:
      - goal_match_counts: { 'SDG 1': #synonyms_matched, 'SDG 2': #synonyms_matched, ... }
      - total_matched: sum of all synonyms matched across all goals
    """
    doc_lower = doc_text.lower()
    
    goal_match_counts = {g: 0 for g in sdg_dict.keys()}
    total_matched = 0
    
    for goal, keywords in sdg_dict.items():
        matched_count = 0
        for kw in keywords:
            if kw in doc_lower:
                matched_count += 1
        goal_match_counts[goal] = matched_count
        total_matched += matched_count
    
    return goal_match_counts, total_matched

def compute_sdg_percentage(doc_text, sdg_dict):
    """
    For a given doc, returns { 'SDG 1': ratio, 'SDG 2': ratio, ... } 
    ratio = (# matched synonyms in that Goal) / (# matched synonyms across all Goals).
    If total_matched=0, ratio=0 for all.
    """
    goal_match_counts, total_matched = match_sdg_keywords(doc_text, sdg_dict)

    sdg_ratios = {}
    if total_matched == 0:
        for g in sdg_dict.keys():
            sdg_ratios[g] = 0.0
    else:
        for g in sdg_dict.keys():
            sdg_ratios[g] = goal_match_counts[g] / total_matched

    return sdg_ratios

###############################################################################
# 3. PROCESSING A SINGLE SPLIT
###############################################################################

from aiohttp import ClientTimeout


def process_eurlex_split(split_name, sdg_dict, output_csv):
    """
    Loads the specified split from the EUR-Lex dataset,
    computes coverage ratios for each doc, and saves to output_csv.

    We include 'celex_id', 'title', 'text' in the output, plus SDG coverage columns.

    :param split_name: "train", "validation", or "test"
    :param sdg_dict: dictionary of { 'SDG X': set_of_synonyms, ... }
    :param output_csv: path to save results for this split
    """
    # Use aiohttp.ClientTimeout for timeout configuration
    storage_options = {
        "client_kwargs": {
            "timeout": ClientTimeout(total=600)  # Timeout set to 10 minutes
        }
    }

    # Pass storage_options and trust_remote_code
    dataset = load_dataset(
        "NLP-AUEB/eurlex",
        split=split_name,
        trust_remote_code=True,
        storage_options=storage_options,
    )
    print(f"Processing {split_name} split: {len(dataset)} documents")

    results = []
    for item in dataset:
        doc_id = item["celex_id"]
        title = item["title"]
        text = item["text"]

        sdg_ratios = compute_sdg_percentage(text, sdg_dict)

        # Build row with celex_id, title, text
        row = {"celex_id": doc_id, "title": title, "text": text}
        # Add coverage columns for each SDG
        for goal in sdg_dict.keys():
            row[goal] = sdg_ratios[goal]
        results.append(row)

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Saved {split_name} results to {output_csv}")


###############################################################################
# 4. MAIN ORCHESTRATION - PROCESS ALL SPLITS
###############################################################################

def main_excel_to_coverage(xlsx_path,
                           out_train="eurlex_sdg_coverage_train.csv",
                           out_val="eurlex_sdg_coverage_val.csv",
                           out_test="eurlex_sdg_coverage_test.csv"):
    """
    1) Loads synonyms from the given Excel file
    2) Processes train, validation, and test splits of EUR-LEX
    3) Saves three CSV files
    """
    # 1. Load synonyms
    sdg_dict = load_sdg_keywords_from_excel(xlsx_path)
    print("Loaded synonyms for", len(sdg_dict), "SDGs")
    
    # 2. Process train
    process_eurlex_split("train", sdg_dict, out_train)
    # 3. Process validation
    process_eurlex_split("validation", sdg_dict, out_val)
    # 4. Process test
    process_eurlex_split("test", sdg_dict, out_test)

###############################################################################
# 5. EXECUTION EXAMPLE
###############################################################################

if __name__ == "__main__":
    excel_path = "term_matrix.xlsx"  # your Excel file with synonyms

    main_excel_to_coverage(
        xlsx_path=excel_path,
        out_train="eurlex_sdg_coverage_train.csv",
        out_val="eurlex_sdg_coverage_val.csv",
        out_test="eurlex_sdg_coverage_test.csv"
    )

Loaded synonyms for 17 SDGs


Generating validation split: 100%|██████████| 6000/6000 [00:00<00:00, 6867.28 examples/s]


Processing train split: 45000 documents
Saved train results to eurlex_sdg_coverage_train.csv
Processing validation split: 6000 documents
Saved validation results to eurlex_sdg_coverage_val.csv
Processing test split: 6000 documents
Saved test results to eurlex_sdg_coverage_test.csv
